# ED Admission Prediction — Survey-Aware Deep Dive Report

Presentation layer over the Survey-Aware Deep Dive (`ML/reports/survey_aware_deep_dive/`) — every table and chart below loads already-computed artifacts; **nothing is recomputed or retrained by running this notebook**.

This deep dive answers the project's core survey-aware research question (`Docs/PROJECT_CONTEXT.md` §44): does accounting for NHAMCS's `PATWT` sample weights change the production LightGBM model's (1) raw performance, (2) SHAP explanations, and (3) fairness across race/ethnicity groups, relative to the standard unweighted model?


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd
from IPython.display import Image, display, Markdown

DEEP_DIVE_DIR = REPO_ROOT / "ML" / "reports" / "survey_aware_deep_dive"


## 1. Performance: Survey-Weighted vs. Unweighted LightGBM

Identical hyperparameters (from `experiment_log.json`, the production tuning result) and identical training data — the only difference is `sample_weight=PATWT` on the weighted variant. This isolates the effect of survey weighting on the actual production model, closing the gap left by Sprint 2's Logistic-Regression-only comparison.

In [2]:
performance = json.loads((DEEP_DIVE_DIR / "weighted_vs_unweighted_metrics.json").read_text())

performance_table = pd.concat(
    {
        split: pd.DataFrame(performance[split])
        for split in ("validation", "test")
    },
    axis=1,
)
performance_table.drop(index="confusion_matrix", errors="ignore")


validation                 test           
              weighted unweighted  weighted unweighted
accuracy      0.932612   0.933028  0.926789   0.930532
precision     0.761745   0.771626  0.735099   0.755932
recall        0.713836   0.701258  0.698113   0.701258
sensitivity   0.713836   0.701258  0.698113   0.701258
specificity   0.965964    0.96836  0.961649   0.965484
f1            0.737013   0.734761  0.716129   0.727569
roc_auc       0.962682   0.964851  0.953424   0.956429
pr_auc        0.814796   0.827476  0.740801   0.759931
brier_score   0.055525   0.052812  0.062581   0.059324
threshold          0.5        0.5       0.5        0.5

## 2. Explanations: SHAP Comparison

Sprint 3's SHAP analysis ran on the unweighted model only — flagged as open "Future Work". Here the survey-weighted model's SHAP values (via the same `TreeExplainer` approach) are compared against the already-computed unweighted SHAP values on the identical validation split.

In [3]:
shap_comparison = json.loads((DEEP_DIVE_DIR / "shap_comparison.json").read_text())
ranking = shap_comparison["ranking_comparison"]

print(f"Spearman rank correlation: {ranking['spearman_rank_correlation']:.4f} "
      f"(p={ranking['spearman_p_value']:.3e})")
print(f"Top-{ranking['top_n']} overlap: {ranking['top_n_overlap_count']}/{ranking['top_n']}")
print(f"Only in unweighted top-{ranking['top_n']}: {ranking['only_in_unweighted_top_n']}")
print(f"Only in weighted top-{ranking['top_n']}: {ranking['only_in_weighted_top_n']}")


Spearman rank correlation: 0.9725 (p=2.598e-248)
Top-20 overlap: 18/20
Only in unweighted top-20: ['DRUGID2', 'SURGDAY']
Only in weighted top-20: ['NUMGIV', 'POOLNURS']


![Weighted vs Unweighted Importance](../reports/survey_aware_deep_dive/figures/weighted_vs_unweighted_importance.png)

### Decision-Flip Rate

Across the full validation split (not just anecdotal examples): how often does survey weighting flip a prediction across the 0.5 decision threshold, and where do those flips concentrate?

In [4]:
flip_stats = shap_comparison["decision_flip_stats"]
pd.Series(flip_stats, name="decision_flip_stats").to_frame()


,decision_flip_stats
flip_count,47.000000
total,2404.000000
flip_rate,0.019551
mean_distance_from_boundary_when_flipped,0.165575
mean_distance_from_boundary_when_not_flipped,0.480737


### Per-Patient Comparison

The same 3 patients examined in Sprint 3 Milestone 4, now compared under both models.

In [5]:
pd.DataFrame(shap_comparison["patient_comparisons"]).T


,row_index,unweighted_probability,weighted_probability,unweighted_top_feature,weighted_top_feature
patient_1,1809,0.999865,0.999063,CONSULT__Yes,CONSULT__Yes
patient_2,1219,0.0,0.0,DIAG1__frequency,PROC__1
patient_3,2213,0.506702,0.439065,CONSULT__Yes,CONSULT__Yes


## 3. Fairness Audit: Race/Ethnicity (RACERETH)

The piece `Docs/PROJECT_CONTEXT.md` §44 explicitly names as a survey-aware evaluation criterion ("Fairness") that had never been assessed before this deep dive. Standard group-fairness metrics — selection rate, true positive rate, false positive rate, and per-group ROC-AUC — computed at the same 0.5 threshold used everywhere else in this project.

In [6]:
fairness = json.loads((DEEP_DIVE_DIR / "fairness_audit.json").read_text())

unweighted_groups = pd.DataFrame(fairness["unweighted_group_metrics"]).T
weighted_groups = pd.DataFrame(fairness["weighted_group_metrics"]).T

group_comparison = pd.concat({"unweighted": unweighted_groups, "weighted": weighted_groups}, axis=1)
group_comparison


unweighted                                                          \
           n actual_admission_rate selection_rate true_positive_rate   
1     1310.0              0.144275       0.126718           0.682540   
2      598.0              0.110368       0.108696           0.681818   
3      394.0              0.106599       0.096447           0.738095   
4      102.0              0.205882       0.196078           0.857143   

                                weighted                                       \
  false_positive_rate   roc_auc        n actual_admission_rate selection_rate   
1            0.033006  0.963567   1310.0              0.144275       0.129008   
2            0.037594  0.951014    598.0              0.110368       0.115385   
3            0.019886  0.980858    394.0              0.106599       0.104061   
4            0.024691  0.977660    102.0              0.205882       0.186275   

                                                    
  true_positive_rate false_positive_rate   roc_auc  
1           0.693122            0.033898  0.961906  
2           0.712121            0.041353  0.945403  
3           0.761905            0.025568  0.979573  
4           0.809524            0.024691  0.981775

![Fairness Selection Rate](../reports/survey_aware_deep_dive/figures/fairness_selection_rate.png)

In [7]:
disparity_comparison = pd.concat(
    {
        "unweighted_gap": pd.DataFrame(fairness["unweighted_disparity"]).loc["gap"],
        "weighted_gap": pd.DataFrame(fairness["weighted_disparity"]).loc["gap"],
    },
    axis=1,
)
disparity_comparison["change"] = disparity_comparison["weighted_gap"] - disparity_comparison["unweighted_gap"]
disparity_comparison


,unweighted_gap,weighted_gap,change
selection_rate,0.099632,0.082214,-0.017418
true_positive_rate,0.175325,0.116402,-0.058923
false_positive_rate,0.017708,0.016662,-0.001046
roc_auc,0.029844,0.036372,0.006528


## 4. Synthesis

Full narrative and caveats: [`summary.md`](../reports/survey_aware_deep_dive/summary.md).

**Bottom line**: survey weighting costs a small amount of raw in-sample discrimination (expected — `PATWT` up-weights underrepresented sampling strata), does not meaningfully change which features drive the model's predictions (Spearman correlation ~0.97), concentrates the predictions it *does* change on cases the model was already uncertain about, and narrows most fairness gaps across race/ethnicity groups. Together these results are the strongest evidence in the project so far that survey-aware learning is not just methodologically correct but practically worthwhile.